# SIMD 编程模型与接口概述

## 前置要求

学习本节前，建议你已经了解以下内容：

- Ascend C 核函数的基本定义和启动方式；
- Global Memory、Local Memory、Vector计算单元、Cube计算单元 等基本硬件概念；
- C/C++ 指针、模板和命名空间的基本用法。

## 本节大纲

本节将围绕 AI Core SIMD 编程模型，学习 SPMD 嵌套 SIMD 编程范式、AI Core 核心计算单元、SIMD 算子通用开发步骤，以及 Ascend C 多级编程接口的定位。

- SPMD 嵌套 SIMD 编程范式；
- AI Core 计算单元与矢量类、矩阵类、融合类算子；
- SIMD 算子 Kernel 的通用开发步骤；
- Memory 矢量计算与 Reg 矢量计算的数据流；
- 语言扩展层SIMD API、基础API、TPipe/TQue框架编程API 的定位差异；

## 1. SPMD 嵌套 SIMD 编程范式

昇腾 NPU 的 AI Core 算子开发可以先理解为两层并行模型：外层是多 AI Core 的 SPMD (Single Program Multiple Data) 并行，内层是单 AI Core 的 SIMD 细粒度并行。多核 SPMD 层面，多个 AI Core 执行同一份 Kernel 代码，通过 `block_idx` 区分不同 Block，并据此划分每个 AI Core 处理的数据范围。单核 SIMD 层面，单个 AI Core 内部使用硬件计算单元批量处理同质数据，根据计算类型调度 Vector 或 Cube，并配合片上本地存储完成计算。因此，SIMD 算子的并行逻辑通常不是为每个数据元素单独写线程，而是先把全局数据切成多个 Block，再由每个 AI Core 在自己的数据分片上完成搬运和批量计算。

## 2. AI Core 计算单元与算子类型

AI Core 可以从控制、计算和存储三个方面理解。标量处理单元负责地址偏移、指令调度和控制流；Vector 计算单元执行向量指令，适合元素级、逻辑类、数据重组类计算；Cube 计算单元面向矩阵乘加、卷积等规整的算力密集型计算；本地存储单元，用于缓存计算中间数据，规避频繁访问低速全局内存的性能损耗，其中Cube计算单元配套 L1 Buffer、L0C Buffer等；Vector计算单元配套统一缓存UB (Unified Buffer)；Ascend 950PR/Ascend 950DT的AI Core向量单元新增可编程向量寄存器（Register）。

基于主要计算单元，AI Core 算子通常可以分为三类：

- 矢量类算子：核心计算逻辑由 Vector 计算单元承载，以元素级计算、逻辑运算和数据重组为主，核函数使用 `__vector__` 修饰；
- 矩阵类算子：核心计算逻辑由 Cube 计算单元承载，以矩阵乘、卷积等规整密集计算为主，核函数使用 `__cube__` 修饰；
- 融合类算子：同一个算子中同时包含矩阵密集计算和向量灵活处理，联动 Cube 与 Vector，核函数使用 `__mix__(cube, vec)` 修饰。

## 3. SIMD 算子 Kernel 的通用开发步骤

AI Core SIMD 编程的一个显著特征是显式分层访存。开发者需要把数据从 Global Memory 搬运到 AI Core 片内 Local Memory，在片内完成计算，再把结果写回 Global Memory。

一个 SIMD 算子 Kernel 通常围绕四个步骤展开：

1. Tiling 分块设计：将全局张量切分为多个数据块，让不同 AI Core 获得相对均衡的数据分片；
2. 数据搬入：将待计算数据从 Global Memory 搬入 L1 Buffer、UB 等片上存储，需要关注搬运粒度、对齐和片上空间占用；
3. 数据计算：调度 Vector 或 Cube 完成向量、矩阵或混合计算，需要处理计算单元选择、数据依赖和同步关系；
4. 数据搬出：将片上计算结果写回 Global Memory，需要关注结果布局、搬出粒度和写回顺序。

由于 AI Core 片上存储空间有限，实际开发中通常会采用分块迭代、分批计算和流水线并行等方式，降低搬运开销对整体性能的影响。

## 4. Memory 矢量计算与 Reg 矢量计算

在矢量计算场景中，可以先区分 Memory 矢量计算和 Reg 矢量计算两种数据流。二者都服务于 Vector 计算，但使用的片上存储层级不同，适用的性能目标也不同。

Memory 矢量计算，具备全架构兼容、流程简洁稳定、通用性强的特点，全程基于UB完成数据缓存与运算，其数据流通常是：
  - 数据搬入：Global Memory → UB
  - 矢量计算：基于 **UB** 完成矢量计算
  - 数据搬出：UB → Global Memory

Reg 矢量计算是 Ascend 950 系列新增的矢量计算模式，它进一步利用寄存器低延迟、高带宽的特点，在 Register 上完成连续矢量计算，减少中间结果反复落 UB 的开销，适合更高性能诉求，其数据流为：
  - 数据搬入：全局内存 → UB → 寄存器（**Reg**）
  - 矢量计算：基于 **Reg** 完成矢量计算
  - 数据搬出：寄存器（**Reg**） → UB → 全局内存

## 5. Ascend C 多级编程接口

Ascend C 面向 AI Core 算子开发提供多级编程接口。它们都可以调度 AI Core 算力，但抽象层次、资源管理方式和适用场景不同。

| 接口层级 | 语言 | 编程对象 | 资源和同步方式 | 适合场景 |
| --- | --- | --- | --- | --- |
| TPipe/TQue框架编程API | C++ | Tensor | 通过 TPipe/TQue 管理内存搬运、队列通信和同步 | 希望框架编排搬运和计算，提高编程易用性 |
| 基础API | C++ | Tensor | 通过 `LocalMemoryAllocator` 等接口分配 Tensor，由开发者自主管理同步和内存布局 | 需要 C++ Tensor 表达，同时保留较强的底层控制能力 |
| 语言扩展层SIMD API | C | 指针 | 通过地址限定符、静态数组和原生接口管理本地内存与同步 | 需要贴近硬件指令和本地存储，追求极致性能与可控性 |

这三层接口是本节理解 SIMD 编程 API 的主线。开发时通常根据开发效率、性能目标、团队熟悉的语言风格以及对底层细节的控制需求来选择。

## 6. 接口选择参考

结合前面的编程模型和接口层级，可以用下面的思路做初步判断。

如果希望框架统一管理队列同步、片上内存生命周期和流水线，优先选择 TPipe/TQue框架编程API。如果希望使用 C++ Tensor 接口，同时自主控制搬运、计算和同步，优先选择基础API。如果需要直接控制指针、本地存储、硬件指令和核内同步，优先选择语言扩展层SIMD API。

接口选择不是固定顺序。一般可以先用更高层的封装快速完成常见逻辑；当性能瓶颈明确、数据流复杂或需要精细控制硬件资源时，再下沉到更底层的接口。对于不确定的场景，可以先回到 AI Core 计算单元和算子类型，判断算子主要属于矢量、矩阵还是融合计算，之后选择API。

## 7. 本节小结

本节围绕 AI Core SIMD 编程模型和 Ascend C 多级编程接口进行了概览。

- AI Core SIMD 算子通常采用外层多核 SPMD 与内层单核 SIMD 的双层并行方式；
- AI Core 内部包含标量处理、Vector、Cube 和本地存储等关键组成；
- SIMD 算子开发通常围绕 Tiling、数据搬入、数据计算、数据搬出四个步骤展开；
- Memory 矢量计算以 UB 为主要缓存和计算载体，Ascend 950 系列新增的 Reg 矢量计算进一步利用寄存器完成连续矢量计算；
- Ascend C 多级编程接口包括 TPipe/TQue框架编程API、基础API、语言扩展层SIMD API；

## 8. 参考资料

- [SIMD 编程模型概述](<https://gitcode.com/cann/asc-devkit/blob/master/docs/guide/编程指南/编程模型/AI-Core-SIMD编程/概述.md>)
- [SIMD API 检索首页](<https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/SIMD-API.md>)

## 9. 课后练习

1. （单选题）AI Core SIMD 算子开发可以先理解为哪两层并行模型？  
   A. 单核串行执行 + 全局同步执行  
   B. 外层多核 SPMD 并行 + 内层单核 SIMD 并行  
   C. 外层 SIMD 并行 + 内层 SPMD 并行  
   D. Host 侧计算并行 + AI Core 侧仅负责数据搬运  

2. （单选题）`block_idx` 在多核 SPMD 编程中的主要作用是什么？  
   A. 表示 Host 侧线程编号  
   B. 表示 Tensor 的数据类型  
   C. 区分不同 Block，用于划分每个 AI Core 处理的数据范围  
   D. 表示 UB 的总容量  

3. （单选题）以下哪一项最符合 AI Core SIMD 编程中的显式分层访存特点？  
   A. 所有数据始终只在 Global Memory 中直接完成计算  
   B. 将数据从 Global Memory 搬入片上本地存储计算，再将结果写回 Global Memory  
   C. 每一步中间结果都必须先写回 Global Memory 后才能继续计算  
   D. Kernel 不需要处理任何数据搬运  

4. （单选题）SIMD 算子 Kernel 的通用开发步骤通常可以概括为哪一项？  
   A. Tiling 分块设计 -> 数据搬入 -> 数据计算 -> 数据搬出  
   B. 数据搬出 -> 数据计算 -> Tiling 分块设计 -> 数据搬入  
   C. 只进行 Tiling 分块设计，不需要处理数据搬运和计算  
   D. 先搬出结果，再根据结果反推数据分块方式  

5. （单选题）本节介绍的 Ascend C 多级编程接口主线包括哪三类？  
   A. TPipe/TQue框架编程API、基础API、语言扩展层SIMD API  
   B. Host API、Device API、Runtime API  
   C. Tiling API、调度 API、执行 API  
   D. 数据搬运 API、同步 API、调试 API  

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.03.03_answer.txt